# WRIME Ver.2の読み込み

Hugging Face DatasetsからWRIME Ver.2を読み込み、公式分割、クラス分布、列構造、先頭データを確認します。

In [ ]:
%pip install -q datasets==2.21.0

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "shunk031/wrime",
    name="ver2",
    trust_remote_code=True,
)
dataset

## 分割ごとのデータ数

In [ ]:
{split_name: len(split) for split_name, split in dataset.items()}

## `writer.sentiment`のクラス分布

各分割について、5クラス（−2、−1、0、+1、+2）の件数と割合を集計します。

In [ ]:
import pandas as pd

LABELS = [-2, -1, 0, 1, 2]

class_counts = pd.DataFrame(
    {
        split_name: (
            pd.Series(
                [writer["sentiment"] for writer in split["writer"]]
            )
            .value_counts()
            .reindex(LABELS, fill_value=0)
        )
        for split_name, split in dataset.items()
    }
)
class_counts.index.name = "writer.sentiment"
class_counts

In [ ]:
class_percentages = class_counts.div(class_counts.sum(axis=0), axis=1) * 100
class_percentages.round(2)

### 不均衡の要約

最多クラスと最少クラスの件数比が1に近いほど均等です。

In [ ]:
imbalance_summary = []

for split_name in class_counts.columns:
    counts = class_counts[split_name]
    majority_label = int(counts.idxmax())
    minority_label = int(counts.idxmin())
    majority_count = int(counts.max())
    minority_count = int(counts.min())

    imbalance_summary.append(
        {
            "split": split_name,
            "最多クラス": majority_label,
            "最多割合(%)": round(majority_count / counts.sum() * 100, 2),
            "最少クラス": minority_label,
            "最少割合(%)": round(minority_count / counts.sum() * 100, 2),
            "最多/最少比": round(majority_count / minority_count, 2),
        }
    )

pd.DataFrame(imbalance_summary).set_index("split")

### 確認結果

- trainは`0`が30.76%で最多、`−2`が11.81%で最少（最多/最少比2.60）。
- validationは`+1`が33.48%で最多、`+2`が11.64%で最少（最多/最少比2.88）。
- testは`+1`が32.72%で最多、`−2`が10.08%で最少（最多/最少比3.25）。

単一クラスが大半を占めるほど極端ではありませんが、3分割とも均等ではなく、無視できないクラス不均衡があります。また、最多クラスがtrainとvalidation・testで異なるため、分割間の分布差にも注意が必要です。

## 列構造

In [ ]:
dataset["train"].features

## 学習用データの先頭1件

In [ ]:
sample = dataset["train"][0]
sample

## 本研究で使用する投稿本文と書き手の感情極性

書き手の感情極性は、`writer.sentiment`という独立した列ではなく、`writer`内の`sentiment`に格納されています。

In [ ]:
print("投稿本文:", sample["sentence"])
print("writer.sentiment:", sample["writer"]["sentiment"])